# Tool 1: RAG_search

In [0]:

spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.rag_search(
    question STRING COMMENT 'The medical or facility question to search for semantically. 
                             Examples: "which hospitals have ICU beds", 
                             "find facilities with dialysis treatment",
                             "what equipment does Banhart Specialist Hospital have",
                             "Are there any clinics in Accra that do heart treatment?",
                             Do NOT use for counting or aggregation queries — use sql_query instead.',

    region   STRING COMMENT 'Optional. Filter results to a specific Ghana region.
                             Pass NULL to search all regions.
                             Valid values: Greater Accra Region, Ashanti Region, 
                             Western Region, Eastern Region, Central Region,
                             Northern Region, Upper East Region, Upper West Region,
                             Volta Region, Bono Region, Oti Region, Ahafo Region,
                             Bono East Region, North East Region, Savannah Region,
                             Western North Region.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Semantic search over Ghana medical facility documents using vector similarity.
         
         USE THIS TOOL WHEN the question involves:
         - What capabilities, procedures, or equipment a facility has
         - Finding facilities that offer a specific medical service
         - Questions about clinical specialties or care levels
         - Free-text descriptions of facility services
         - Any question where the answer requires understanding meaning not exact keywords
         
         DO NOT USE THIS TOOL FOR:
         - Counting facilities (use sql_query instead)
         - Filtering by structured fields like city or type (use sql_query instead)
         - Getting full details of one specific facility (use get_facility instead)
         
         RETURNS: JSON string with fields:
         - answer: natural language answer based on retrieved facilities
         - citations: list of facilities used, each with name, city, region, 
                      facility_type, lat, lon, relevance_score
         - mappable: facilities with GPS coordinates for map display
         - num_sources: number of facilities retrieved'
AS $$
import requests
import json

WORKSPACE_URL = "your-WORKSPACE_URL"
RAG_ENDPOINT  = "your-RAG_ENDPOINT"
DATABRICKS_TOKEN="your-DATABRICKS_TOKEN"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}

payload = {
    "dataframe_records": [
        {
            "question": question, 
            "region"  : region if region else None
        }
    ]
}

try:
    resp = requests.post(
        f"https://{WORKSPACE_URL}/serving-endpoints/{RAG_ENDPOINT}/invocations",
        headers = headers,
        json    = payload,
        timeout = 180
    )
    if resp.status_code != 200:
        return json.dumps({"error": f"RAG failed: {resp.status_code}: {resp.text}"})

    pred      = resp.json()["predictions"][0]
    citations = pred.get("citations", "[]")
    mappable  = pred.get("mappable_facilities", "[]")

    return json.dumps({
        "answer"     : pred.get("answer", ""),
        "citations"  : json.loads(citations) if isinstance(citations, str) else citations,
        "mappable"   : json.loads(mappable)  if isinstance(mappable,  str) else mappable,
        "num_sources": pred.get("num_sources", 0)
    })

except Exception as e:
    return json.dumps({"error": str(e)})
$$
""")

print("rag_search UC Function created successfully")

# Tool 2: Genie Text2SQL

In [0]:
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.sql_query(
    question STRING COMMENT 'Plain English question that requires structured data analysis.
                             Examples:
                             "how many hospitals are in Greater Accra Region",
                             "How many hospitals in Accra have the ability to perform treatments related to the Heart ?",
                             "How many hospitals have cardiology?",
                             "list all public hospitals in Ashanti Region",
                             "which region has the most healthcare facilities",
                             "find regions with fewer than 3 hospitals",
                             Be specific — include filters, groupings, or conditions 
                             you want in the SQL.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Converts plain English to SQL and runs it against the Ghana facilities Delta Table.
         
         USE THIS TOOL WHEN the question involves:
         - Counting: how many, total number of
         - Filtering: list all X in Y, facilities where Z
         - Aggregating: average, sum, maximum, minimum
         - Ranking: which region has most/least, top N
         - Comparing numbers across regions or facility types
         - Any question that needs exact numbers from structured columns
                  
         RETURNS: JSON string with fields:
         - answer: natural language description of the result
         - sql_query: the SQL that was executed
         - results: list of result rows as dicts (max 20 rows)
         - num_rows: total number of rows returned'
AS $$
import requests
import json
import time

WORKSPACE_URL = "your-WORKSPACE_URL"
GENIE_SPACE_ID = "your-genie-space-id"
DATABRICKS_TOKEN="your-DATABRICKS_TOKEN"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

BASE = f"https://{WORKSPACE_URL}/api/2.0/genie/spaces/{GENIE_SPACE_ID}"

try:
    # STEP 1: Start conversation
    start = requests.post(
        f"{BASE}/start-conversation",
        headers=headers,
        json={"content": question},
        timeout=30
    )

    if start.status_code != 200:
        return json.dumps({"error": f"Genie failed: {start.text}"})

    data = start.json()
    conv_id = data["conversation_id"]
    msg_id = data["message_id"]

    # STEP 2: Poll
    message = None
    for _ in range(30):
        poll = requests.get(
            f"{BASE}/conversations/{conv_id}/messages/{msg_id}",
            headers=headers,
            timeout=30
        )

        message = poll.json()
        status = message.get("status", "")

        if status == "COMPLETED":
            break
        elif status in ("FAILED", "CANCELLED"):
            return json.dumps({"error": f"Genie {status}", "raw": message})

        time.sleep(2)

    # STEP 3: Parse attachments
    attachments = message.get("attachments", [])

    sql_query = None
    description = None
    text_answer = ""

    for a in attachments:
        if "query" in a:
            sql_query = a["query"].get("query")
            description = a["query"].get("description")

        if "text" in a:
            text_answer += a["text"].get("content", "") + " "

    text_answer = text_answer.strip()

    # If no SQL, return text
    if not sql_query:
        return json.dumps({
            "answer": text_answer or "No answer returned",
            "sql_query": None,
            "results": [],
            "num_rows": 0
        })

    # STEP 4: Get results
    result_resp = requests.get(
        f"{BASE}/conversations/{conv_id}/messages/{msg_id}/query-result",
        headers=headers,
        timeout=60
    )

    if result_resp.status_code != 200:
        return json.dumps({"error": result_resp.text})

    data = result_resp.json()

    columns = [
        c["name"]
        for c in data.get("statement_response", {})
                     .get("manifest", {})
                     .get("schema", {})
                     .get("columns", [])
    ]

    rows = data.get("statement_response", {}) \
               .get("result", {}) \
               .get("data_typed_array", [])

    results = []
    for r in rows:
        values = [v.get("str", None) for v in r.get("values", [])]
        results.append(dict(zip(columns, values)))

    return json.dumps({
        "answer": text_answer or description or f"{len(results)} rows returned",
        "sql_query": sql_query,
        "results": results[:20],
        "num_rows": len(results)
    })

except Exception as e:
    return json.dumps({"error": str(e)})

$$
""")

print("sql_query function created")

# 3 Get Facility Details

In [0]:
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.get_facility(
    name STRING COMMENT 'Name of the specific facility to look up.
                         Can be exact or partial name.
                         Examples: "2BN Military Hospital", "Abuakwa Maternity Home".
                         Use when you already know the facility name from 
                         a previous rag_search or sql_query result.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Fetch complete structured details for one specific Ghana medical facility by name.
         
         USE THIS TOOL WHEN:
         - You found a facility name from rag_search and need its exact structured data
         - You need to verify specific claims about a named facility
         - You need the exact lat/lon of a specific facility for geospatial calculations
         - You need to cross-reference a facility found in one tool with structured data
         - A user asks specifically about one named facility
         
         DO NOT USE THIS TOOL FOR:
         - Searching across multiple facilities (use rag_search instead)
         - Counting or aggregating (use sql_query instead)
         
         RETURNS: JSON string with fields:
         - found: true/false
         - facility: dict with all columns for this facility
         - lat: latitude as float (0.0 if unknown)
         - lon: longitude as float (0.0 if unknown)'
AS $$
import json

import requests, time

WORKSPACE_URL = "your-WORKSPACE_URL"
GENIE_SPACE_ID = "your-genie-space-id"
DATABRICKS_TOKEN="your-DATABRICKS_TOKEN"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

BASE = f"https://{WORKSPACE_URL}/api/2.0/genie/spaces/{GENIE_SPACE_ID}"

question = (
    f"Get all details for the facility named '{name}'. "
    f"Return name, facilityTypeId, operatorTypeId, address_city, "
    f"address_stateOrRegion, numberDoctors, capacity, lat, lon, "
    f"specialties, capability, description."
)

try:
    start   = requests.post(f"{BASE}/start-conversation",
                            headers=headers, json={"content": question}, timeout=30)
    
    if start.status_code != 200:
        return json.dumps({"error": f"Failed to start: {start.text}"})
        
    conv_id = start.json()["conversation_id"]
    msg_id  = start.json()["message_id"]
    message = None

    for _ in range(30):
        poll   = requests.get(f"{BASE}/conversations/{conv_id}/messages/{msg_id}",
                              headers=headers, timeout=30)
        message = poll.json()
        status = message.get("status")
        
        if status == "COMPLETED":
            break
        elif status in ("FAILED", "CANCELLED"):
            return json.dumps({"error": f"Genie query failed: {status}"})
            
        time.sleep(2)
    
    attachments = message.get("attachments", [])
    sql_query = None

    for a in attachments:
        if "query" in a:
            sql_query = a["query"].get("query")

    # FIX #2: Return correct schema if no SQL
    if not sql_query:
        return json.dumps({
            "found": False,
            "facility": {},
            "error": "No SQL generated"
        })

    # STEP 4: Get results
    result_resp = requests.get(
        f"{BASE}/conversations/{conv_id}/messages/{msg_id}/query-result",
        headers=headers,
        timeout=60
    )

    if result_resp.status_code != 200:
        return json.dumps({"error": result_resp.text})

    data = result_resp.json()

    columns = [
        c["name"]
        for c in data.get("statement_response", {})
                     .get("manifest", {})
                     .get("schema", {})
                     .get("columns", [])
    ]

    rows = data.get("statement_response", {}) \
               .get("result", {}) \
               .get("data_typed_array", [])

    results = []
    for r in rows:
        values = [v.get("str", None) for v in r.get("values", [])]
        results.append(dict(zip(columns, values)))

    # FIX #1: Correct Indentation
    if results:
        f = results[0]
        
        # FIX #3: Safely convert to float
        try:
            lat = float(f.get("lat")) if f.get("lat") else 0.0
        except (ValueError, TypeError):
            lat = 0.0
            
        try:
            lon = float(f.get("lon")) if f.get("lon") else 0.0
        except (ValueError, TypeError):
            lon = 0.0

        return json.dumps({
            "found"   : True,
            "facility": f,
            "lat"     : lat,
            "lon"     : lon
        })
        
    return json.dumps({"found": False, "facility": {}})

except Exception as e:
    return json.dumps({"error": str(e)})

$$
""")
print("created get_facility function")
